<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Build leakage-resistant, reproducible training and inference code with explicit data and model contracts.</p>
</div>

## Learning objectives

- Separate training, validation, test, and inference responsibilities.
- Prevent leakage by fitting transformations only on training data.
- Build reproducible pipelines with controlled randomness.
- Define inference schemas, evaluation outputs, and model metadata.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## Training and inference are different systems

Training consumes labeled historical data and produces a fitted artifact plus metrics and metadata. Inference consumes new features and returns predictions under a stable schema. Reusing one transformation pipeline is essential, but training-only operations—label access, resampling, validation splits—must never enter serving code.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PredictionRequest:
    age: float
    monthly_usage: float
    support_tickets: int

    def as_features(self) -> list[float]:
        if self.age < 0 or self.monthly_usage < 0 or self.support_tickets < 0:
            raise ValueError("features cannot be negative")
        return [self.age, self.monthly_usage, float(self.support_tickets)]


request = PredictionRequest(age=32, monthly_usage=18.5, support_tickets=2)
print(request.as_features())


## Leakage and reproducibility

Leakage occurs when training uses information unavailable at prediction time or learns preprocessing from validation/test data. Split first, then fit imputers, encoders, scalers, and feature selectors on training data only. Record random seeds, data version, feature schema, dependency versions, parameters, and metrics.


In [ ]:
experiment = {
    "experiment_id": "churn-2026-09-09-01",
    "random_seed": 42,
    "data_version": "customers-v3",
    "target": "churned",
    "features": ["age", "monthly_usage", "support_tickets"],
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "primary_metric": "roc_auc",
}

assert sum(experiment["split"].values()) == 1.0
print(experiment)


## Evaluation is a decision contract

A metric is useful only in the context of error costs, class balance, and an operating threshold. Preserve per-slice results for important populations and time periods. A model card should state intended use, exclusions, data, metrics, limitations, and ownership.


In [ ]:
def classification_counts(actual: list[int], predicted: list[int]) -> dict:
    if len(actual) != len(predicted):
        raise ValueError("actual and predicted lengths differ")
    tp = sum(a == 1 and p == 1 for a, p in zip(actual, predicted))
    tn = sum(a == 0 and p == 0 for a, p in zip(actual, predicted))
    fp = sum(a == 0 and p == 1 for a, p in zip(actual, predicted))
    fn = sum(a == 1 and p == 0 for a, p in zip(actual, predicted))
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn}


print(classification_counts([1, 0, 1, 0], [1, 1, 0, 0]))


## Worked example: fit/transform protocol without leakage

The scaler learns only from training values and reuses those parameters for validation and inference.


In [ ]:
from dataclasses import dataclass
import numpy as np


@dataclass
class StandardScaler:
    mean_: np.ndarray | None = None
    scale_: np.ndarray | None = None

    def fit(self, values: np.ndarray):
        self.mean_ = values.mean(axis=0)
        std = values.std(axis=0)
        self.scale_ = np.where(std == 0, 1, std)
        return self

    def transform(self, values: np.ndarray) -> np.ndarray:
        if self.mean_ is None or self.scale_ is None:
            raise RuntimeError("fit must be called before transform")
        return (values - self.mean_) / self.scale_


train = np.array([[20, 5], [30, 10], [40, 15]], dtype=float)
validation = np.array([[50, 20]], dtype=float)
scaler = StandardScaler().fit(train)
print(scaler.transform(train))
print(scaler.transform(validation))


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Write separate training and inference input contracts.
2. List five leakage examples and the prevention rule for each.
3. Build a simple transformer with `fit` and `transform` methods.
4. Create model metadata containing data version, feature order, seed, metrics, threshold, and limitations.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
model_metadata = {
    "model_name": "learner-completion-risk",
    "model_version": "1.0.0",
    "data_version": "enrollments-2026-09-01",
    "feature_order": ["attendance_rate", "assessment_average", "days_inactive"],
    "random_seed": 42,
    "metrics": {"roc_auc": 0.86, "recall_at_threshold": 0.78},
    "decision_threshold": 0.35,
    "intended_use": "prioritize optional learner support outreach",
    "limitations": [
        "not a measure of learner ability",
        "requires monitoring for cohort drift",
        "human review required before outreach",
    ],
}

required = {"model_version", "data_version", "feature_order", "metrics", "limitations"}
if missing := required - model_metadata.keys():
    raise ValueError(f"incomplete model metadata: {sorted(missing)}")
print(model_metadata)


## Knowledge check

**1. When should preprocessing be fitted?**

::: {.callout-note collapse="true"}
### Answer
On training data only.
:::

**2. Why preserve feature order?**

::: {.callout-note collapse="true"}
### Answer
Many models interpret numeric arrays positionally.
:::

**3. Is a high aggregate metric enough?**

::: {.callout-note collapse="true"}
### Answer
No; evaluate error costs, thresholds, slices, stability, and intended use.
:::


## Recap

- Split before fitting transformations.
- Version the full data-to-prediction contract.
- Evaluate models as decision systems, not isolated scores.


<div class="lesson-nav">
<a href="17-concurrency-and-performance.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> Concurrency, Performance, and Memory</a>
<a href="19-capstone-learning-analytics-pipeline.html">Capstone: Learning Analytics Data Product <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
